# ARE-006.1: recommendations for a YummyAnime profile

This notebook only loads the saved ARE-006 model. No training, tuning, dataset downloads or validation/test evaluation. Notebook 04 contains training and validation research.

Install the existing research/model extras and configure the public application token in local .env (YUMMYANIME_APPLICATION_TOKEN). Never use a private token or owner Bearer token. Histories and nicknames stay in memory. Clear all outputs and execution counts before saving; Jupyter may autosave outputs while you work.

In [ ]:
import os
from IPython.display import display
from anime_recommender.exploration import project_root
from anime_recommender.recommendation_exploration import (
    load_profile_workspace, optional_profile_view, profile_display, refresh_profile_recommendations, enrich_profile_titles,
)

root = project_root()

## Nickname and display parameters

Environment variable wins. Otherwise enter a nickname; Enter uses First_Light. Rerun this cell to enter another nickname (unset the environment override if needed). The entered value is never written to a project file. top_k controls recommendation count; show_history controls the optional private-history display.

In [ ]:
nickname = (
    os.getenv("YUMMYANIME_TEST_NICKNAME", "").strip()
    or input("Enter your YummyAnime nickname: ").strip()
    or "First_Light"
)

top_k = 15
show_history = False

## Load saved models

Run once per kernel. No train/validation/test interaction files are loaded. Missing saved artifacts produce a preparation instruction: prepare ARE-005.1 data and run python -m anime_recommender.training.pipeline separately, then rerun this cell. This notebook never trains a model.

In [ ]:
workspace = load_profile_workspace(root)

## Fetch and recommend

Rerun after changing nickname. For a changed top_k, rerun only the Top-N cell using the fetched history. At most three public API calls: profile, lists (including user ratings and MAL IDs), favorites. No separate rating-statistics request is needed. Failure stops inference rather than using a partial history.

Binary positives: completed with no rating (0/missing), rating>=5 or favorite. Planned/dropped veto positivity. Watching alone is not positive; an already rated/favorite watching item follows the same existing policy. No new weights. MAL mapping identifies warm positives for fixed-item ALS fold-in. All known mapped items are excluded; empty warm context uses Popularity fallback.

In [ ]:
profile = optional_profile_view(workspace, nickname=nickname, top_k=top_k, env_path=root / ".env")
view = profile_display(profile, nickname=nickname, show_history=show_history)
display(profile["requests"])
display(view["summary"].to_frame("Value"))

## Top-N

ALS scores are dot products, not probabilities. Coverage is reported both against the full local catalog and the train vocabulary. Excluded candidate items counts known train items removed from the candidate set. Unmapped items cannot be identified by dataset ID. Recommendations may be fewer than K if too few candidates remain.

Russian titles use verified local language fields or one bounded YummyAnime catalog lookup by exact MAL ID (up to 20 IDs, no pagination). Results stay in memory and are reused when changing K. Missing, ambiguous, non-Russian or failed lookups show a dash. No automatic translation. The English column prefers explicit English fields, then the original catalog title (which may be romanized). For Popularity fallback, ALS Score shows a dash; the selected model remains in the summary.

YummyAnime links are omitted: the local training catalog has no certified YummyAnime URL/ID mapping for recommended items. MAL IDs are shown without inventing YummyAnime URLs.

In [ ]:
profile = refresh_profile_recommendations(workspace, profile, top_k=top_k)
profile = enrich_profile_titles(profile, env_path=root / ".env")
view = profile_display(profile, nickname=nickname, show_history=show_history)
display(view["recommendations"])
display(profile.get("title_error"))

## Optional history inspection

Set show_history=True in the parameter cell, then rerun this cell to inspect statuses, original ratings, favorite flags, positives and mapping. This does not repeat API calls or fold-in. Do not save the displayed personal data in notebook outputs.

In [ ]:
view = profile_display(profile, nickname=nickname, show_history=show_history)
display(view["history"])